# 开放数据集上的质检、覆盖和四级标注

还没有自有头戴设备时，用带手部位姿的开放数据把规格 v10 的前半跑通：**统一 episode → QC/产出率 → 覆盖空档**，校验四级标注，再把 QC 通过的片段导出成 LeRobot v3.0。动作是多步手腕增量。线性 BC 在固定的 episode 验证集上打出 `val_loss` 和「保持当前手腕」基线；有 GPU 时可以用同一份数据跑 lerobot 0.6.1 的 ACT。差距分析见 `docs/gap_analysis.md`，定义见 `docs/headcam_data_spec.md` §8。

默认只跑**合成的 EgoDex 式 HDF5**（两条短片段），不下载数据。真实 EgoDex 测试集 `test.zip` 约 16.1GB，许可 CC BY-NC-ND，不要提交进仓库。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skyxiaobai/robot/blob/main/notebooks/open_dataset_pipeline_colab.ipynb)


In [ ]:
%pip install -q numpy h5py pyarrow pandas matplotlib pillow


In [ ]:
import os
import subprocess
from pathlib import Path

root = Path.cwd()
if not (root / "scripts" / "demo_open_dataset_pipeline.py").exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/skyxiaobai/robot.git", "/content/robot",
    ])
    os.chdir("/content/robot")
print("repo", Path.cwd())


## 合成样本

一条手腕在画面中央移动，一条置信度很低且手腕在画面外。产出率应约为 50%：只有通过片段级 QC 的帧计入可用数据。两条都缺 SUBTASK 和 INSTRUCTION，因为源数据只有整段描述。


In [ ]:
import subprocess
import sys
from pathlib import Path

out = Path("outputs/open_data_demo")
subprocess.check_call([
    sys.executable, "scripts/demo_open_dataset_pipeline.py",
    "--out", str(out),
])
print((out / "yield_episodes.csv").read_text(encoding="utf-8"))
print((out / "coverage_counts.csv").read_text(encoding="utf-8"))


## 看一条样本

`scripts/ego_visualize.py` 用和质检相同的投影，把 21 点手骨架、手腕轨迹和任务文字画到画面上，并排比较通过与各拒绝原因，再画头和双手腕的世界系轨迹。合成样本没有 mp4，就画在灰底上。真实数据把同名 mp4 放在 HDF5 旁边，或给 `overlay` 传 `--video`。图写在 `outputs/`，不要提交。


In [ ]:
import subprocess
import sys
from pathlib import Path

unified = Path("outputs/open_data_demo/unified")
viz = Path("outputs/open_data_demo/viz")
episodes = sorted(unified.rglob("*.json"))
if not episodes:
    raise SystemExit("先跑上面的合成样本")
sample = episodes[0]
subprocess.check_call([
    sys.executable, "scripts/ego_visualize.py", "overlay",
    "--episode", str(sample),
    "--out", str(viz / "overlay.png"),
    "--frame", "5", "--width", "640",
])
subprocess.check_call([
    sys.executable, "scripts/ego_visualize.py", "overlay",
    "--episode", str(sample),
    "--out", str(viz / "overlay.mp4"),
    "--width", "640", "--limit-frames", "12",
])
subprocess.check_call([
    sys.executable, "scripts/ego_visualize.py", "qc_compare",
    "--episodes", str(unified),
    "--out", str(viz / "qc_compare.png"),
])
subprocess.check_call([
    sys.executable, "scripts/ego_visualize.py", "traj3d",
    "--episode", str(sample),
    "--out", str(viz / "traj3d.png"),
    "--frame", "5",
])
print("wrote", viz)


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "scripts/validate_hierarchy.py",
    "examples/hierarchy_annotation.json", "--strict",
])


## 换成 EgoDex 测试集抽样

只在磁盘够的时候做。下载和解压都不在默认单元里。

```bash
curl -L "https://ml-site.cdn-apple.com/datasets/egodex/test.zip" -o test.zip
unzip test.zip
python scripts/demo_open_dataset_pipeline.py --input test --out outputs/egodex_sample --limit 20
```

`--limit 20` 只转换前 20 条 HDF5。转换结果里的世界系是 ARKit 原点，不是本仓库重新做的 SLAM。严格校验会失败，直到有人补上时间分段子任务和分手指令。pusht 上的 ACT 仍走 `notebooks/act_pusht_colab.ipynb`。同一份产出率 CSV 可以接下面的 LeRobot 导出：把 `--episodes` 和 `--yield-csv` 换成 `outputs/egodex_sample/unified` 与 `outputs/egodex_sample/yield_episodes.csv`。


## 导出 LeRobot 并做几步线性 BC

接上面合成演示的 `outputs/open_data_demo`。只导出产出率 CSV 里 `accepted=yes` 的片段。合成样本没有 mp4，导出会写 16×16 占位视频（需要系统里的 ffmpeg，Colab 默认有）。

`observation.state` 是双手关节加当前手腕（140 维）。`action` 每一行是**下一步手腕增量**（dxyz + 相对四元数，14 维），不是下一帧的绝对位姿。线性 BC 把连续 16 行拼成动作块。验证集按 episode 留出，各档数据量共用；这里只有一条通过 QC 的合成片段，脚本会改留该条末尾的固定样本作验证。`--max-frames` 是训练动作块的条数上限：先在每条 episode 里按种子打乱块的起点，再在各条之间轮转抽取，直到凑满。更小的上限是同一顺序的前缀，不是按整段 episode 往上加。日志里的 `copy_current_wrist` 是「增量全为保持不动」的基线，缩放律只读 `val_loss`。真实 mp4 默认缩到 224×224。

In [ ]:
import subprocess
import sys
from pathlib import Path

out = Path("outputs/open_data_demo")
dataset = Path("outputs/egodex_lerobot")
subprocess.check_call([
    sys.executable, "scripts/ego_to_lerobot.py",
    "--episodes", str(out / "unified"),
    "--yield-csv", str(out / "yield_episodes.csv"),
    "--out", str(dataset),
    "--horizon", "16",
    "--video-size", "224",
])
log_all = Path("outputs/ego_pretrain_all.log")
log_small = Path("outputs/ego_pretrain_8.log")
subprocess.check_call([
    sys.executable, "scripts/ego_pretrain_bc.py",
    "--dataset", str(dataset), "--steps", "5",
    "--log", str(log_all),
])
subprocess.check_call([
    sys.executable, "scripts/ego_pretrain_bc.py",
    "--dataset", str(dataset), "--steps", "5",
    "--max-frames", "8",
    "--log", str(log_small),
])
print(log_small.read_text(encoding="utf-8"))
print(log_all.read_text(encoding="utf-8"))

In [ ]:
import subprocess
import sys
from pathlib import Path

def _frames(text):
    for line in text.splitlines():
        if line.startswith("policy="):
            for part in line.split():
                if part.startswith("frames="):
                    return part.split("=", 1)[1]
    raise SystemExit("日志里没有 frames=")

log_all = Path("outputs/ego_pretrain_all.log")
log_small = Path("outputs/ego_pretrain_8.log")
runs = Path("outputs/ego_pretrain_runs.yaml")
runs.write_text(
    "runs:\n"
    "  - name: ego_small\n"
    "    size: %s\n"
    "    log: ego_pretrain_8.log\n"
    "  - name: ego_all\n"
    "    size: %s\n"
    "    log: ego_pretrain_all.log\n" % (
        _frames(log_small.read_text(encoding="utf-8")),
        _frames(log_all.read_text(encoding="utf-8")),
    ),
    encoding="utf-8",
)
html = Path("outputs/ego_scaling.html")
subprocess.check_call([
    sys.executable, "scripts/scaling_law.py",
    "--runs", str(runs),
    "--out", str(html),
])
print(html.read_text(encoding="utf-8")[:400])

## Colab GPU 上的 ACT

线性 BC 只是快速基线。要在同一份导出上训 ACT，用 **lerobot 0.6.1** 的 `lerobot-train`：数据集每行动作是一步手腕增量，`policy.chunk_size=16` 预测接下来 16 步。真实视频在导出时用 `--video-size 224`（或 256）缩放，不要按 1080p 重编码。

pusht 上的 ACT 仍走 `notebooks/act_pusht_colab.ipynb`，这里不改它的 action[2]。下面默认不跑：把 `RUN_ACT` 改成 `True`，并选 GPU 运行时。合成占位视频只能确认命令能启动；真正预训练把 `--dataset.root` 指到带 224 视频的导出目录。配方也写在 `examples/ego_act_train.yaml`。

In [ ]:
import subprocess
import sys
from pathlib import Path

RUN_ACT = False  # GPU 运行时改成 True
DATASET = Path("outputs/egodex_lerobot")
OUTPUT = Path("outputs/ego_act_smoke")

if not RUN_ACT:
    print("跳过 ACT。要跑时把 RUN_ACT 改成 True。命令见 examples/ego_act_train.yaml")
elif not DATASET.joinpath("meta", "info.json").is_file():
    raise SystemExit("先跑上面的导出单元")
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "lerobot[training]==0.6.1",
    ])
    if OUTPUT.exists():
        raise SystemExit(f"{OUTPUT} 已存在。lerobot-train 在 resume=false 时不会覆盖它。")
    subprocess.check_call([
        "lerobot-train",
        "--dataset.repo_id=local/egodex",
        f"--dataset.root={DATASET.resolve()}",
        "--policy.type=act",
        "--policy.device=cuda",
        "--policy.push_to_hub=false",
        "--policy.chunk_size=16",
        "--policy.n_action_steps=16",
        "--policy.n_obs_steps=1",
        "--batch_size=2",
        "--steps=50",
        "--save_freq=50",
        "--log_freq=10",
        "--num_workers=0",
        "--seed=1000",
        "--wandb.enable=false",
        "--policy.optimizer_lr=1e-5",
        "--policy.optimizer_lr_backbone=1e-5",
        f"--output_dir={OUTPUT.resolve()}",
        "--job_name=egodex_act_smoke",
    ])